# Vault Collection — geoid-only retrieval (auto-DENY edition)

**Use case**: a collection where features are persisted but unreachable without the geoid returned at write time. The geoid acts as a capability token. No filter search, no listing, no spatial query, no tiles, no STAC `/search` — losing the geoid means losing the feature.

## What pre-#480 looked like

Earlier revisions of this notebook hand-rolled an IAM bundle: a stack of regex-scoped DENY policies, paired ALLOW carve-outs, a `vault-{cat}` Role, and a dedicated test principal bound to that role. The bundle existed because pinning `items_elasticsearch_private_driver` on a collection's routing did **not** automatically install the catalog-wide DENY — `_apply_deny_policy` only fired from the driver's `ensure_storage()` (cold provision) and `_restore_deny_policies()` (cold boot).

## What changed in #480

`_on_apply_items_routing_config` now calls `_sync_deny_policy_for_catalog` after every items-routing-config write. The transitions are:

| Previous routing | New routing | Action |
|---|---|---|
| public | private | `_apply_deny_policy(catalog_id)` — install `private_deny_{cat}` |
| private | public, catalog still has private collections | no-op (DENY stays) |
| private | public, no private collection left | `_revoke_deny_policy(catalog_id)` — drop the policy |
| public | public | no-op |

Bound to the implicit `all_users` role, the auto-DENY blocks `GET` on every OGC service prefix under `/{ogc}/catalogs/{cat}/...`. The pattern is built from the `OGCServiceMixin` contributor registry (PR #479), so it self-maintains as new OGC protocols come online.

Net effect for this notebook: the manual bundle, the vault Role, and the dedicated test principal all disappear. We instead **flip-public-to-private** on an existing collection, read the auto-policy back, and probe enforcement against anonymous callers.

## Routing shape

| Operation | Driver(s) | Purpose |
|---|---|---|
| **WRITE** | `items_postgresql_driver` (sync, fatal) + `items_elasticsearch_private_driver` (sync, warn) | PG = authoritative geometry; ES = geoid-keyed index for fast lookup |
| **READ** | `items_postgresql_driver` (`hints={"geometry_exact"}`) | Authoritative un-simplified geometry |
| **SEARCH** | `items_elasticsearch_private_driver` | Geoid lookup only — driver capabilities exclude `SPATIAL_FILTER`/`ATTRIBUTE_FILTER`/`FULLTEXT` |

## Two retrieval modes

1. **Always-available — accurate geometry**: `GET /features/catalogs/{cat}/collections/{coll}/items/{geoid}` → READ driver = PG with `geometry_exact` hint → un-simplified. **Blocked by the auto-DENY once private** — exercise it before the flip, or with a principal carrying an ALLOW override scoped to the items-by-id path.
2. **Optional — fast cross-collection lookup**: `GET /search/catalogs/{cat}/geoid/{geoid}` → ES private index. `/search` is not an `OGCServiceMixin` contributor, so the auto-DENY pattern does **not** include it. The route stays reachable to whoever holds direct ALLOW on `/search`.

In [ ]:
import os
import asyncio
import httpx

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=False)
BASE = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080").rstrip("/")

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass
KEYCLOAK = os.environ.get("KEYCLOAK_TOKEN_URL", "http://localhost:8180/realms/geoid/protocol/openid-connect/token")
CATALOG_ID = "demo-vault"
COLLECTION_ID = "vault-locations"

TOKEN = (
    os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or os.environ.get("DYNASTORE_TOKEN")
    or ""
)
if not TOKEN:
    print("No env token — fetching one from Keycloak (testadmin)…")
    async with httpx.AsyncClient(timeout=15.0) as kc:
        r = await kc.post(KEYCLOAK, data={
            "grant_type": "password",
            "client_id": "geoid-api",
            "client_secret": "geoid-api-secret",
            "username": "testadmin",
            "password": "testpassword",
        })
    if r.status_code == 200:
        TOKEN = r.json().get("access_token", "")
        print(f"Got token from Keycloak (len={len(TOKEN)})")
    else:
        print(f"Keycloak token request failed: {r.status_code} — {r.text[:200]}")

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"
client = httpx.AsyncClient(base_url=BASE, headers=headers, timeout=120.0)

# Anonymous client — no Authorization header. Used to probe the auto-DENY
# enforcement surface in step 6 without needing to mint a per-principal token.
anon_client = httpx.AsyncClient(base_url=BASE, headers={"Content-Type": "application/json"}, timeout=60.0)


def _check(r, label=""):
    status = r.status_code
    ok = status < 400
    body = ""
    try:
        body = r.text[:300]
    except Exception:
        pass
    print(f"{label + ': ' if label else ''}{status}{'' if ok else f' — {body}'}")
    return r


# Probe whether the `search` extension is mounted on this deployment.
# `/search/catalogs/{any}/geoid/{any}` is the canonical fast-lookup route
# but it's only available when SCOPE includes `search`. Without it, the
# notebook uses the always-available items-by-id path instead.
try:
    _probe = await client.get("/search/catalogs/_probe_/geoid/00000000-0000-0000-0000-000000000000")
    SEARCH_AVAILABLE = _probe.status_code in (200, 400, 404)
except Exception:
    SEARCH_AVAILABLE = False

print(f"BASE             = {BASE}")
print(f"TOKEN            = {'set (' + str(len(TOKEN)) + ' chars)' if TOKEN else 'EMPTY (writes will fail)'}")
print(f"SEARCH ext       = {'mounted (fast-lookup available)' if SEARCH_AVAILABLE else 'absent (will use items-by-id only)'}")

## 1. Create a public catalog + public collection (clean slate)

We deliberately start fully public. The auto-DENY only fires when an items routing-config write flips the private-driver presence false→true — so a fresh provision (which goes through `ensure_storage`'s own apply hook) and a routing-write flip both demonstrate the contract. This notebook exercises the routing-write path.

In [ ]:
_cleanup = await client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"}, timeout=60.0)
if _cleanup.status_code not in (204, 404):
    print(f"WARN: cleanup returned {_cleanup.status_code}: {_cleanup.text[:200]}")

r = await client.post("/stac/catalogs", json={
    "id": CATALOG_ID,
    "title": "Vault Demo",
    "description": "Vault collections — geoid-only retrieval, no discovery.",
})
_check(r, "Catalog")

for _ in range(60):
    r = await client.get(f"/stac/catalogs/{CATALOG_ID}")
    if r.status_code == 200 and r.json().get("provisioning_status") == "ready":
        print("Catalog ready")
        break
    await asyncio.sleep(1)
else:
    print("WARN: catalog still provisioning after 60s")

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLLECTION_ID,
    "title": "Vault Locations",
    "description": "Sensitive geometries reachable only by geoid.",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
    },
})
_check(r, "Collection")

## 2. Pre-flip baseline: no `private_deny_{cat}` exists

The catalog was just provisioned without any private-driver pin, so the auto-DENY should not be present. We list catalog-scoped policies and check explicitly. (`GET /admin/policies` returns the policy list; there is no GET-single endpoint as of #911 — `Principal` consolidation — so we filter client-side.)

In [ ]:
DENY_POLICY_ID = f"private_deny_{CATALOG_ID}"

async def _get_deny_policy():
    """Return the auto-DENY policy dict or None. Catalog-scoped policies
    live in the catalog partition (`?catalog_id={cat}`); the platform-tier
    list does not include them."""
    r = await client.get("/admin/policies", params={"catalog_id": CATALOG_ID})
    if r.status_code != 200:
        return None
    for p in r.json() or []:
        if p.get("id") == DENY_POLICY_ID:
            return p
    return None

baseline = await _get_deny_policy()
assert baseline is None, (
    f"Expected no `{DENY_POLICY_ID}` before the routing flip, found: {baseline}"
)
print(f"OK — no `{DENY_POLICY_ID}` policy present pre-flip.")

## 3. Flip public → private via `collection_routing_config`

PUT the routing config that pins `items_elasticsearch_private_driver` in WRITE (fan-out alongside PG) and SEARCH, with READ pinned to PG + `geometry_exact`. The apply-phase handler (`_on_apply_items_routing_config`) detects the false→true transition and calls `ItemsElasticsearchPrivateDriver._apply_deny_policy(catalog_id)`.

> **Routing waterfall reminder**: collection-scoped routing config **merges** with catalog → platform → code defaults rather than replacing them. After the PUT, GET-back will show SEARCH still includes other drivers (e.g., `items_postgresql_driver`, `items_duckdb_driver`) inherited from the platform default. The private ES driver's lack of `SPATIAL_FILTER`/`ATTRIBUTE_FILTER`/`FULLTEXT` makes filter/spatial queries structurally inert at the driver layer, but the route stays open at the routing layer. **Real lockdown of catalog-wide GETs comes from the auto-DENY**, verified in steps 4 and 6.

In [ ]:
routing = {
    "enabled": True,
    "operations": {
        "WRITE": [
            {"driver_ref": "items_postgresql_driver",
             "on_failure": "fatal", "write_mode": "sync"},
            {"driver_ref": "items_elasticsearch_private_driver",
             "on_failure": "warn",  "write_mode": "sync"},
        ],
        "READ": [
            {"driver_ref": "items_postgresql_driver",
             "hints": ["geometry_exact"]},
        ],
        "SEARCH": [
            {"driver_ref": "items_elasticsearch_private_driver"},
        ],
    },
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/collection_routing_config",
    json=routing,
)
_check(r, "PUT collection_routing_config (flip pub→priv)")

r = await client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/collection_routing_config",
)
ops = r.json().get("operations", {})
print("\nApplied routing.operations:")
for op, entries in ops.items():
    summary = [f"{e.get('driver_ref')}({','.join(e.get('hints') or []) or '-'})" for e in (entries or [])]
    print(f"  {op:8s} {summary}")

## 4. Assert the auto-DENY was installed by the apply handler

After PR #1142ee81 the apply-phase handler on `ItemsRoutingConfig` calls `_sync_deny_policy_for_catalog`, which in turn calls `_apply_deny_policy` when the routing now pins the private items driver and didn't before. The policy is registered against the implicit `all_users` role so every principal evaluates it.

The resource pattern is built from `get_ogc_service_prefixes()` — every loaded `OGCServiceMixin` contributor — so the pattern shape is `^/({prefix1}|{prefix2}|…)/catalogs/{cat}(/.*)?$`. We don't pin the exact prefix list (it varies by SCOPE) but we do pin shape invariants.

In [ ]:
import re

deny = await _get_deny_policy()
assert deny is not None, f"Expected `{DENY_POLICY_ID}` to be auto-installed after routing flip."

# Shape invariants — keep these stable regardless of which OGC extensions
# are mounted in this SCOPE.
assert deny["effect"] == "DENY", deny
assert "GET" in deny["actions"], deny
assert len(deny["resources"]) >= 1, deny
pattern = deny["resources"][0]
assert f"catalogs/{CATALOG_ID}" in pattern, f"DENY pattern must scope to {CATALOG_ID!r}: {pattern}"

# Compile the pattern to confirm it actually matches the surfaces we expect.
rx = re.compile(pattern)
must_match = [
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    f"/stac/catalogs/{CATALOG_ID}/search",
]
for path in must_match:
    if rx.search(path):
        print(f"  DENY covers: {path}")
    else:
        print(f"  NOTE: pattern does not match {path} — prefix may be missing in this SCOPE")

print(f"\nauto-DENY policy {DENY_POLICY_ID!r}:")
print(f"  effect    = {deny['effect']}")
print(f"  actions   = {deny['actions']}")
print(f"  resources = {pattern}")

## 5. Write 2 features — small + large geometry

WRITE fan-out goes through PG (fatal-sync) + private ES (warn-sync). The auto-DENY only blocks `GET`, so writes by the sysadmin continue to work. ES has a 10MB per-doc soft limit; the private driver simplifies geometries above the threshold via `simplify_to_fit` (`elasticsearch.py:460`). PG is unaffected.

In [ ]:
import math

SMALL = {
    'type': 'Feature',
    'geometry': {'type': 'LineString', 'coordinates': [[0.0, 0.0], [0.1, 0.1]]},
    'properties': {'label': 'small', 'sensitivity': 'low'},
}

N = 5000
ring = [
    [10.0 + 0.5 * math.cos(2 * math.pi * i / N), 20.0 + 0.5 * math.sin(2 * math.pi * i / N)]
    for i in range(N)
]
ring.append(ring[0])
LARGE = {
    'type': 'Feature',
    'geometry': {'type': 'Polygon', 'coordinates': [ring]},
    'properties': {'label': 'large', 'sensitivity': 'high', 'vertex_count': N},
}

fc = {'type': 'FeatureCollection', 'features': [SMALL, LARGE]}
r = await client.post(
    f'/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items',
    json=fc,
)
_check(r, 'Write 2 features (sysadmin POST — unaffected by GET-only auto-DENY)')

body = r.json() if r.status_code < 400 else {}
ids = body.get('ids') or body.get('accepted_ids') or []
print('\nAssigned geoids (capability tokens — keep them safe!):')
for g in ids:
    print(f'  {g}')
GEOIDS = ids

## 6. Verify enforcement — anonymous probes every OGC service prefix

We hit the `/admin/policies` list to recover the same OGC prefix set the auto-DENY was built from, then probe each one as an anonymous caller. Each should return `403` (or `401` on deployments that gate IAM via authentication first). The auto-DENY is bound to the `all_users` role so every principal — including the unauthenticated one — evaluates it.

Because the auto-DENY is `GET`-only and pinned to OGC service prefixes, two surfaces explicitly stay open:

- `POST /stac/catalogs/{cat}/search` (POST not in the actions set).
- `GET /search/catalogs/{cat}/geoid/{geoid}` (`/search` is not an `OGCServiceMixin`).

In [ ]:
# Recover the prefix list from the policy pattern itself, so this cell stays
# in sync with whichever OGC extensions are mounted on this deployment.
m = re.match(r'^/(?:\^/)?\(([^)]+)\)/catalogs/', pattern) or re.match(r'^/?\(([^)]+)\)/catalogs/', pattern)
if m:
    PREFIXES = [p.replace('\\', '') for p in m.group(1).split('|')]
else:
    PREFIXES = []
    print(f"WARN: could not parse prefixes from pattern {pattern!r}; falling back to a literal probe set")
    PREFIXES = ['features', 'stac', 'records', 'tiles', 'edr', 'coverages', 'consys', 'maps']

print(f"Probing {len(PREFIXES)} OGC service prefixes anonymously:")
denied = 0
for px in sorted(PREFIXES):
    path = f'/{px}/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items'
    r = await anon_client.get(path)
    blocked = r.status_code in (401, 403)
    flag = 'BLOCKED' if blocked else f'OPEN ({r.status_code})'
    print(f'  GET {path:80s} → {flag}')
    if blocked:
        denied += 1

print(f'\n{denied}/{len(PREFIXES)} OGC GETs blocked under the auto-DENY pattern.')
assert denied == len(PREFIXES), 'auto-DENY did not block all enumerated OGC service prefixes anonymously'

# Explicit ALLOW carve-out: /search is not an OGC contributor.
if SEARCH_AVAILABLE and GEOIDS:
    r = await anon_client.get(f'/search/catalogs/{CATALOG_ID}/geoid/{GEOIDS[0]}')
    print(f'\n  GET /search/catalogs/{CATALOG_ID}/geoid/<known>  → {r.status_code}'
          f'  (auto-DENY does not cover /search; anonymous reach depends on the route\'s own gating)')

## 7. Flip private → public; assert auto-DENY revoked

PUT a public-only items routing back. The apply handler re-runs `_sync_deny_policy_for_catalog`; this time the new routing has no private items driver and `_catalog_has_private_collection` returns False (we only have one collection in the catalog, and we just made it public). So `_revoke_deny_policy` deletes `private_deny_{cat}`.

If we had two private collections and flipped only one, the catalog scan would still find a private sibling and the DENY would stay (this is the asymmetric-revoke transition pinned in `test_sync_does_not_revoke_when_other_private_collection_remains`).

In [ ]:
public_routing = {
    "enabled": True,
    "operations": {
        "WRITE": [
            {"driver_ref": "items_postgresql_driver",
             "on_failure": "fatal", "write_mode": "sync"},
        ],
        "READ": [
            {"driver_ref": "items_postgresql_driver"},
        ],
        "SEARCH": [
            {"driver_ref": "items_postgresql_driver"},
        ],
    },
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/collection_routing_config",
    json=public_routing,
)
_check(r, "PUT collection_routing_config (flip priv→pub)")

post_flip = await _get_deny_policy()
assert post_flip is None, (
    f"Expected `{DENY_POLICY_ID}` to be revoked after the last private "
    f"collection went public, but it still exists: {post_flip}"
)
print(f"OK — `{DENY_POLICY_ID}` was revoked when the catalog lost its last private collection.")

# Sanity: anonymous probe should no longer 403 on the OGC surfaces. Some
# routes may still 401/403 for other reasons (IAM defaults gate listing),
# but the catalog-wide DENY specifically is gone — listing the policy
# returns empty for this id.
r = await anon_client.get(f'/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items')
print(f"  anon GET /features/.../items after revoke → {r.status_code}  "
      f"(no longer DENY-by-private; whatever you see now is the deployment\'s default IAM)")

## 8. Cleanup

In [ ]:
# Best-effort cleanup so re-running the notebook is idempotent.
# Every step is guarded so a partial upstream failure still tears down
# every artifact this notebook owns. Running it twice is harmless.
# Running it without any upstream cell executed (e.g. straight after
# kernel restart) is also harmless — the `globals()` guards turn missing
# names into no-ops instead of NameError.

g = globals()


async def _safe(coro_factory, label):
    try:
        r = await coro_factory()
        ok = getattr(r, 'status_code', None)
        print(f'  cleanup {label:55s} -> {ok}')
    except Exception as e:
        print(f'  cleanup {label:55s} -> ERR: {e!r}')


_client = g.get('client')
_anon = g.get('anon_client')
if _client is None:
    print('No `client` in scope — setup cell never ran. Nothing to clean up.')
else:
    _cat = g.get('CATALOG_ID')

    # Drop the catalog (cascades to collections + items). The auto-DENY,
    # if still present, is reaped by the same delete path — deleting the
    # last private collection in the catalog triggers `_revoke_deny_policy`
    # on its own apply handler path, and dropping the catalog removes the
    # partition that holds it either way.
    if _cat:
        await _safe(
            lambda: _client.delete(
                f'/stac/catalogs/{_cat}', params={'force': 'true'}, timeout=60.0,
            ),
            f'catalog {_cat}',
        )

    try:
        await _client.aclose()
    except Exception:
        pass

if _anon is not None:
    try:
        await _anon.aclose()
    except Exception:
        pass

print('Done.')

## Evaluation & recommendations

### Why IAM + routing rather than a driver flag or a new schema field

- **Driver layer is the wrong place**. A driver-side gate would have to be duplicated in PG, ES public, ES private, DuckDB, Iceberg, BigQuery — and would still not cover catalog-scoped routes (tiles, EDR, coverages) where no per-collection driver is consulted. Drivers are platform-agnostic by design; putting access policy there violates the SSOT.
- **A new schema field** (`discovery_mode: "vault"`) was rejected in favour of implicit detection from the routing shape. The presence of `items_elasticsearch_private_driver` in any operation slot uniquely identifies private — the catalog cascade validator enforces the rest.
- **IAM is the correct chokepoint**. Single enforcement point at `IamMiddleware.dispatch`; regex DENY wins on equal-priority ties (PR #866, #915); `partition_key` scopes the policy to the catalog so a catalog admin can self-serve.

### Why the DENY auto-fires now (post-#480)

Pre-#480 the DENY only landed on cold provisioning (`ensure_storage`) and cold boot (`_restore_deny_policies`). Flipping an existing collection's items routing to pin the private driver on a running deployment installed the driver but not the policy — every OGC GET stayed reachable until the next restart. PR #1142ee81 closed that hole by registering `_on_apply_items_routing_config` as the apply handler on `ItemsRoutingConfig` and routing both apply and (symmetric) revoke through `_sync_deny_policy_for_catalog`. The unit suite pins all three live transitions in `tests/dynastore/modules/storage/test_routing_write_auto_deny.py`.

### Cross-extension impact matrix

| Extension | Discovery routes | Covered by auto-DENY |
|---|---|---|
| Features | `/items`, `/queryables` | yes (collection-scoped GETs under `/features/catalogs/{cat}/`) |
| STAC     | `/search` (GET), `/collections-search` (GET), `/items` | yes (POST search NOT covered — action set is GET-only) |
| Records  | `/items` | yes |
| Tiles    | `/{z}/{x}/{y}.mvt` | yes (catalog-scoped — see deep dive below) |
| EDR      | `/position`, `/area`, `/cube`, `/locations` | yes |
| Coverages | `/coverage`, `/domainset`, `/rangetype` | yes |
| Connected Systems | `/systems`, `/datastreams`, `/observations` | yes |
| WFS / Maps | `*` | yes |
| Search   | `/search/catalogs/{cat}` | **no** — `/search` is not an `OGCServiceMixin` contributor |
| Search   | `/geoid/{geoid}` | **no** — same. The geoid lookup stays open by design. |
| Features (items-by-id) | `/items/{geoid}` | yes (GET under `/features/catalogs/{cat}/...`) |

The items-by-id GET being blocked is intentional under the strict-vault contract: once private, the only retrieval path is `/search/.../geoid/{geoid}` — the geoid carries its own capability semantics. If a deployment needs the items-by-id path open to specific principals, add a per-principal ALLOW policy at a higher priority for `^/features/catalogs/{cat}/collections/[^/]+/items/[^/]+/?$`.

### Tile-route deep dive

Tiles are catalog-scoped (no `{collection_id}` path segment). The auto-DENY covers `/tiles/catalogs/{cat}(/.*)?` so a mixed catalog (one private collection, several public) blocks tiles for **every** collection in the catalog. Operational guideline: **one private collection per catalog** when tile-leak avoidance is a hard requirement. The alternative — handler-side filter that skips collections whose routing shape pins the private driver — is filed as a follow-up.

### Recommendations (filed as follow-up issues)

1. **`/search/catalogs/{cat}/geoid/{geoid}?source=geometry_exact`** — fold the two-call accurate-retrieval pattern into a single endpoint by routing the lookup through PG when the hint is requested.
2. **Per-handler vault filter for catalog-scoped routes** (tiles / EDR / coverages / maps / WFS / connected_systems) — so vault and public collections can coexist in one catalog without falling back to the dedicated-catalog guideline.
3. **HATEOAS suppression in parent listings** — private collection IDs still appear in `list_collections_in_catalog`. Not a leak by itself (every GET against the collection 403s) but a tightening opportunity.
4. **Explicit ALLOW preset for items-by-id under private** — a documented policy template for the common case where the strict vault contract is too tight.

### Threat-model notes

- The **geoid is a bearer token**. Treat it like a secret — never log, transmit only over secure channels, rotate by `DELETE`+re-`POST` (a new geoid is assigned on every write).
- The bogus-geoid endpoint returning 404 (vs 403) is an intentional information-leak boundary — it confirms the endpoint is open without disclosing whether the token is valid. Acceptable since a uniform-random UUID is exhaustively unguessable.
- If a deployment also wires a public ES indexer for this catalog, vault features will leak there. The routing config is the SSOT — this notebook deliberately omits public ES, and an audit job should check every private catalog's routing config periodically.